# 🔭 EDA — MaNGA spectra (10k FITS) + morphology catalog (CSV)

**Data**
| Source | Content | Key columns |
|---|---|---|
| ~10,000 `.fits` files (Google Drive) | 1 integrated spectrum per galaxy | `WAVE`, `FLUX_1RE`, `FLUX_FoV`, `ERROR_1RE`, `ERROR_FoV` |
| Large `.csv` catalog | 1 row per galaxy | `name`, `objra`, `objdec`, `Type`, `Ttype`, `nsa_z`, `LogMass`, `incl`, `PETRO_TH90` |

Join key: `name` (e.g. `manga-10001-12701` = plate 10001, IFU design 12701 → 127-fiber bundle).

**Workflow**
1. Configuration & setup → 2. Copy FITS from Drive to local disk → 3. Inventory → 4. Inspect one FITS
5. Catalog EDA → 6. Feature extraction (parallel, cached) → 7. Cross-match
8. Spectral-quality EDA → 9. Spectra vs. morphology → 10. PCA / embeddings → 11. Anomaly detection → 12. Data-quality report & exports

> ⚙️ Edit **only the Configuration cell**, then `Runtime ▸ Run all`. Heavy steps are cached to Drive, so re-runs are fast.

## 0. Configuration

In [ ]:
# ======================= EDIT THESE ===========================================
DRIVE_ROOT   = '/content/drive/MyDrive/manga'          # project folder in Drive
FITS_SOURCE  = f'{DRIVE_ROOT}/fits'                    # folder with .fits / .fits.gz  (or a .zip / .tar file)
CATALOG_CSV  = f'{DRIVE_ROOT}/morfologia.csv'          # large morphology catalog
OUT_DIR      = f'{DRIVE_ROOT}/eda_outputs'             # cached features, figures, reports

LOCAL_FITS   = '/content/fits_local'                   # fast local copy of the FITS files
WORK_DIR     = '/content/work'                         # local scratch (memmaps)

NAME_REGEX   = r'manga-\d+-\d+'                        # how to get `name` from the file name
FITS_COLS    = dict(wave='WAVE', flux_1re='FLUX_1RE', flux_fov='FLUX_FoV',
                    err_1re='ERROR_1RE', err_fov='ERROR_FoV')
SENTINELS    = [-9999, -9999.0, -999, -99]             # values to treat as missing in the catalog
Z_MAX        = 0.20                                    # max redshift used to define the rest-frame grid
DLAM         = 1.5                                     # Å, pixel size of the common grids
N_WORKERS    = None                                    # None → all CPU cores
MAX_FILES    = None                                    # e.g. 300 for a quick dry run; None → all
FORCE_REBUILD = False                                  # True → ignore cached features
RANDOM_STATE = 42
# ==============================================================================

## 1. Setup

In [ ]:
%pip -q install astropy umap-learn

In [ ]:
import os, re, glob, json, time, shutil, warnings, zipfile, tarfile
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.max_columns', 60); pd.set_option('display.width', 200)
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams.update({'figure.dpi': 110, 'figure.figsize': (9, 5)})

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

N_WORKERS = N_WORKERS or os.cpu_count()
for d in (OUT_DIR, f'{OUT_DIR}/figures', LOCAL_FITS, WORK_DIR):
    os.makedirs(d, exist_ok=True)

def savefig(name):
    plt.savefig(f'{OUT_DIR}/figures/{name}.png', bbox_inches='tight')

print(f'Colab: {IN_COLAB} | CPU cores: {os.cpu_count()} | workers: {N_WORKERS}')

## 2. Copy FITS files from Drive to local disk
Reading 10k small files through the Drive mount is **very** slow (each open is a network call).
Copying once to the Colab VM is usually 10–50× faster. *Fastest option:* zip the folder in Drive and point `FITS_SOURCE` to the `.zip`.

In [ ]:
def stage_fits(src, dst):
    t0 = time.time()
    if os.path.isfile(src) and src.endswith('.zip'):
        shutil.copy(src, '/content/_fits.zip')
        with zipfile.ZipFile('/content/_fits.zip') as z: z.extractall(dst)
    elif os.path.isfile(src) and re.search(r'\.tar(\.gz)?$|\.tgz$', src):
        with tarfile.open(src) as t: t.extractall(dst)
    elif os.path.isdir(src):
        if shutil.which('rsync'):
            os.system(f'rsync -a --include="*/" --include="*.fits*" --exclude="*" "{src}/" "{dst}/"')
        else:
            shutil.copytree(src, dst, dirs_exist_ok=True)
    else:
        raise FileNotFoundError(src)
    print(f'Staged in {time.time()-t0:.0f}s')

already = len(glob.glob(f'{LOCAL_FITS}/**/*.fits*', recursive=True))
if already == 0:
    stage_fits(FITS_SOURCE, LOCAL_FITS)
else:
    print(f'{already} FITS files already on local disk — skipping copy')

## 3. File inventory

In [ ]:
files = sorted(glob.glob(f'{LOCAL_FITS}/**/*.fits*', recursive=True))
if MAX_FILES: files = files[:MAX_FILES]

def name_from_path(p):
    m = re.search(NAME_REGEX, os.path.basename(p))
    return m.group(0) if m else Path(p).name.split('.')[0]

inv = pd.DataFrame({'path': files})
inv['name']    = inv.path.map(name_from_path)
inv['size_kb'] = inv.path.map(os.path.getsize) / 1024
inv['matched_regex'] = inv.path.map(lambda p: re.search(NAME_REGEX, os.path.basename(p)) is not None)

print(f'Files: {len(inv):,} | total {inv.size_kb.sum()/1024**2:.2f} GB')
print(f'Names not matching NAME_REGEX: {(~inv.matched_regex).sum()}')
dups = inv[inv.name.duplicated(keep=False)]
print(f'Duplicate names: {dups.name.nunique()}')
display(inv.size_kb.describe().to_frame('size_kb').T)

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(inv.size_kb, bins=60); ax.set(xlabel='File size [kB]', ylabel='N files', title='FITS file sizes')
savefig('01_file_sizes'); plt.show()
# Very small / very large files are often truncated downloads or a different product type
display(inv.nsmallest(5, 'size_kb')[['name', 'size_kb']])

## 4. Inspect one FITS file
Always check the HDU layout and header units before writing the bulk loop. The reader below looks for the first table HDU
that contains the `WAVE` column and supports both layouts: *one row per pixel* and *one row with array columns*.

In [ ]:
from astropy.io import fits

def read_spectrum(path):
    # Return {wave, flux_1re, flux_fov, err_1re, err_fov} as float64 arrays
    with fits.open(path, memmap=False) as hdul:
        for hdu in hdul:
            cols = getattr(hdu, 'columns', None)
            if cols is not None and FITS_COLS['wave'].upper() in [c.upper() for c in cols.names]:
                data = hdu.data
                break
        else:
            raise KeyError(f"No HDU with column {FITS_COLS['wave']}")
        out = {}
        for key, col in FITS_COLS.items():
            a = np.asarray(data[col])
            if a.ndim == 2 and a.shape[0] == 1:      # single-row, vector-column layout
                a = a[0]
            out[key] = a.astype(np.float64).ravel()  # also fixes big-endian dtypes
    return out

with fits.open(files[0]) as h:
    h.info()
    for i, hdu in enumerate(h):
        print(f'\n--- HDU {i} header (first 40 cards) ---')
        print(repr(hdu.header[:40]))

s0 = read_spectrum(files[0])
dw = np.diff(s0['wave'])
print(f"\n{len(s0['wave'])} pixels | λ {s0['wave'].min():.1f}–{s0['wave'].max():.1f} Å | Δλ median {np.median(dw):.3f} Å "
      f"(std {dw.std():.2e}) → {'linear' if dw.std() < 1e-3 else 'non-linear (log?)'} grid")
display(pd.DataFrame(s0).describe().T)

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
for ax, k in zip(axes, ['1re', 'fov']):
    ax.plot(s0['wave'], s0[f'flux_{k}'], lw=.6, label=f'FLUX_{k.upper()}')
    ax.fill_between(s0['wave'], s0[f'flux_{k}'] - s0[f'err_{k}'], s0[f'flux_{k}'] + s0[f'err_{k}'],
                    color='C1', alpha=.3, label='±1σ')
    ax.legend(loc='upper right')
axes[1].set_xlabel('Observed wavelength [Å]'); fig.suptitle(inv.name.iloc[0])
savefig('02_example_spectrum'); plt.show()

## 5. Catalog EDA (large CSV)
Loaded with the multithreaded **pyarrow** CSV engine and compact dtypes. Sentinel values (e.g. `nsa_z = -9999`) become `NaN`.
For very large catalogs, plots use a random sample of up to `PLOT_SAMPLE` rows; statistics use all rows.

In [ ]:
PLOT_SAMPLE = 200_000
CAT_COLS = ['name', 'objra', 'objdec', 'Type', 'Ttype', 'nsa_z', 'LogMass', 'incl', 'PETRO_TH90']
NUM_COLS = ['objra', 'objdec', 'Ttype', 'nsa_z', 'LogMass', 'incl', 'PETRO_TH90']

t0 = time.time()
cat_raw = pd.read_csv(CATALOG_CSV, usecols=CAT_COLS, engine='pyarrow')
print(f'Loaded {len(cat_raw):,} rows × {cat_raw.shape[1]} cols in {time.time()-t0:.1f}s '
      f'| {cat_raw.memory_usage(deep=True).sum()/1024**2:.0f} MB')
missing_cols = set(CAT_COLS) - set(cat_raw.columns)
assert not missing_cols, f'Missing columns: {missing_cols}'

cat = cat_raw.copy()
for c in NUM_COLS:
    cat[c] = pd.to_numeric(cat[c], errors='coerce')
sentinel_counts = {c: int(cat[c].isin(SENTINELS).sum()) for c in NUM_COLS}
cat[NUM_COLS] = cat[NUM_COLS].mask(cat[NUM_COLS].isin(SENTINELS))
cat[NUM_COLS] = cat[NUM_COLS].astype('float32').astype({'objra': 'float64', 'objdec': 'float64'})
cat['Type'] = cat['Type'].astype('category')
print('Sentinel values replaced by NaN:', {k: v for k, v in sentinel_counts.items() if v})
display(cat.head())
display(cat.describe(percentiles=[.01, .25, .5, .75, .99]).T)

In [ ]:
# ---- Derived columns ---------------------------------------------------------
parts = cat['name'].str.extract(r'manga-(\d+)-(\d+)')
cat['plate']     = pd.to_numeric(parts[0], errors='coerce').astype('Int32')
cat['ifudesign'] = pd.to_numeric(parts[1], errors='coerce').astype('Int32')
cat['ifu_size']  = (cat['ifudesign'] // 100).astype('Int16')        # 19, 37, 61, 91, 127 fibers

VALID_TTYPE = (-6, 10)                                               # outside → special code (e.g. 11)
cat['ttype_special'] = cat.Ttype.notna() & ~cat.Ttype.between(*VALID_TTYPE)
cat['Ttype_clean']   = cat.Ttype.where(~cat.ttype_special)

def morph_class(t):
    if pd.isna(t): return 'Unclassified'
    if t <= -4:    return 'E'
    if t <= 0:     return 'S0/S0a'
    if t <= 3:     return 'Sa–Sb'
    return 'Sbc–Irr'
MORPH_ORDER = ['E', 'S0/S0a', 'Sa–Sb', 'Sbc–Irr', 'Unclassified']
cat['morph_class'] = pd.Categorical(cat.Ttype_clean.map(morph_class), MORPH_ORDER, ordered=True)

typ = cat.Type.astype(str)
cat['bar'] = np.select([typ.str.startswith('SAB'), typ.str.startswith('SB')], ['weak', 'strong'], 'none')

# Physical size R90 [kpc] (Planck18 cosmology)
from astropy.cosmology import Planck18
z_ok = cat.nsa_z.where(cat.nsa_z > 0)
kpc_per_arcsec = Planck18.kpc_proper_per_arcmin(z_ok.fillna(0.01).values).value / 60
cat['R90_kpc'] = np.where(z_ok.notna(), cat.PETRO_TH90 * kpc_per_arcsec, np.nan)
cat['cos_incl'] = np.cos(np.deg2rad(cat.incl))

cat_plot = cat.sample(min(len(cat), PLOT_SAMPLE), random_state=RANDOM_STATE)
print(cat.morph_class.value_counts(dropna=False).to_string())

In [ ]:
# ---- Data-quality checks -----------------------------------------------------
checks = {
    'duplicate name':                 cat.name.duplicated(keep=False),
    'name not manga-PLATE-IFU':       cat.plate.isna(),
    'RA outside [0,360)':             ~cat.objra.between(0, 360, inclusive='left'),
    'Dec outside [-90,90]':           ~cat.objdec.between(-90, 90),
    'nsa_z missing/sentinel':         cat.nsa_z.isna(),
    'nsa_z <= 0':                     cat.nsa_z <= 0,
    'LogMass missing':                cat.LogMass.isna(),
    'LogMass outside [7,13]':         cat.LogMass.notna() & ~cat.LogMass.between(7, 13),
    'incl missing':                   cat.incl.isna(),
    'incl == 0 (placeholder?)':       cat.incl == 0,
    'incl outside [0,90]':            cat.incl.notna() & ~cat.incl.between(0, 90),
    'PETRO_TH90 <= 0 or missing':     ~(cat.PETRO_TH90 > 0),
    'Ttype special code':             cat.ttype_special,
    'Type missing':                   cat.Type.isna(),
}
qc_cat = pd.DataFrame({'n': {k: int(v.sum()) for k, v in checks.items()}})
qc_cat['pct'] = (100 * qc_cat.n / len(cat)).round(3)
display(qc_cat[qc_cat.n > 0].sort_values('n', ascending=False))

# Same sky position under different names → repeat observations of one galaxy (≈0.4" rounding)
pos_key = cat.objra.round(4).astype(str) + '_' + cat.objdec.round(4).astype(str)
rep = cat[pos_key.duplicated(keep=False)]
print(f'Rows sharing coordinates with another row: {len(rep):,} ({rep.name.nunique():,} names)')

# Is the Type → Ttype mapping consistent?
tt_map = cat.groupby('Type', observed=True).Ttype.agg(['nunique', 'min', 'max', 'size'])
print('\nTypes mapped to more than one Ttype:'); display(tt_map[tt_map['nunique'] > 1])

In [ ]:
# ---- Missingness -------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
(cat[CAT_COLS].isna().mean() * 100).sort_values().plot.barh(ax=axes[0], color='C3')
axes[0].set(xlabel='% missing', title='Missing values per column')
miss = cat[NUM_COLS].isna()
sns.heatmap(miss.astype(int).corr(), ax=axes[1], cmap='Reds', vmin=0, vmax=1, annot=True, fmt='.2f')
axes[1].set_title('Do columns go missing together?')
savefig('03_missingness'); plt.show()

In [ ]:
# ---- Morphology --------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
cat.Type.value_counts().head(30).plot.bar(ax=axes[0]); axes[0].set_title('Hubble Type (top 30)')
cat.Ttype.value_counts().sort_index().plot.bar(ax=axes[1], color='C2'); axes[1].set_title('T-type (raw, incl. special codes)')
pd.crosstab(cat.morph_class, cat.bar).plot.bar(stacked=True, ax=axes[2]); axes[2].set_title('Morph class × bar')
for ax in axes: ax.tick_params(axis='x', rotation=70)
savefig('04_morphology_counts'); plt.show()

In [ ]:
# ---- Numeric distributions ---------------------------------------------------
plot_cols = ['nsa_z', 'LogMass', 'incl', 'cos_incl', 'PETRO_TH90', 'R90_kpc']
fig, axes = plt.subplots(2, 3, figsize=(16, 7))
for ax, c in zip(axes.ravel(), plot_cols):
    sns.histplot(cat_plot[c].dropna(), bins=60, ax=ax, kde=True)
    ax.set_title(f'{c}  (skew {cat[c].skew():.2f})')
plt.tight_layout(); savefig('05_numeric_hist'); plt.show()
print('Note: for random orientations cos(incl) should be ~uniform; spikes reveal default/placeholder values.')

In [ ]:
# ---- Sky coverage (Mollweide) -------------------------------------------------
ra = np.deg2rad(cat_plot.objra.values); ra = np.where(ra > np.pi, ra - 2*np.pi, ra)
fig = plt.figure(figsize=(12, 6)); ax = fig.add_subplot(111, projection='mollweide')
sc = ax.scatter(-ra, np.deg2rad(cat_plot.objdec), c=cat_plot.nsa_z, s=2, cmap='viridis', vmax=cat.nsa_z.quantile(.99))
ax.set_xticklabels(['150°','120°','90°','60°','30°','0°','330°','300°','270°','240°','210°'])
plt.colorbar(sc, ax=ax, shrink=.6, label='nsa_z'); ax.set_title('Sky distribution (RA increasing to the left)')
savefig('06_sky'); plt.show()

In [ ]:
# ---- Bivariate relations -----------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
hb = axes[0, 0].hexbin(cat_plot.nsa_z, cat_plot.LogMass, gridsize=60, bins='log', mincnt=1, cmap='viridis')
axes[0, 0].set(xlabel='nsa_z', ylabel='LogMass', title='Mass vs redshift (selection function)'); plt.colorbar(hb, ax=axes[0, 0])
sns.violinplot(data=cat_plot, x='morph_class', y='LogMass', ax=axes[0, 1], cut=0, order=MORPH_ORDER)
axes[0, 1].set_title('Stellar mass by morphology')
sns.boxplot(data=cat_plot, x='ifu_size', y='PETRO_TH90', ax=axes[1, 0], showfliers=False)
axes[1, 0].set_title('Galaxy angular size vs IFU bundle size')
sns.histplot(data=cat_plot, x='cos_incl', hue='morph_class', element='step', stat='density',
             common_norm=False, bins=30, ax=axes[1, 1], hue_order=MORPH_ORDER)
axes[1, 1].set_title('Inclination by morphology (ellipticals look round)')
plt.tight_layout(); savefig('07_catalog_bivariate'); plt.show()

display(pd.crosstab(cat.ifu_size, cat.morph_class, normalize='index').round(3))

In [ ]:
# ---- Robust outliers (|robust z| > 5, MAD-based) -----------------------------
def robust_z(s):
    med = s.median(); mad = (s - med).abs().median() * 1.4826
    return (s - med) / mad if mad > 0 else s * np.nan

rz = pd.DataFrame({c: robust_z(cat[c]) for c in ['nsa_z', 'LogMass', 'PETRO_TH90', 'R90_kpc']})
out_mask = (rz.abs() > 5)
print(out_mask.sum().to_string())
display(cat.loc[out_mask.any(axis=1), ['name', 'Type', 'nsa_z', 'LogMass', 'PETRO_TH90', 'R90_kpc']].head(20))

## 6. Feature extraction from all FITS files (parallel + cached)
Each file is read **once** and reduced to:
* **Quality metrics** (both apertures): fraction of bad pixels, negative flux, median S/N, error-array variability, λ coverage.
* **Rest-frame physics** (uses `nsa_z` from the catalog): Dn4000, Hδ_A, EW(Hα, Hβ, [OIII]5007, [NII]6584), optical colour proxy.
* **Spectra resampled to a common rest-frame grid**, normalised at 5400–5600 Å, stored as `float32` memmaps (`.npy`) for stacking and PCA.

Results are cached in `OUT_DIR`, so this step runs only once.

In [ ]:
# ---- Common grids -----------------------------------------------------------
WMIN, WMAX = np.nanmin(s0['wave']), np.nanmax(s0['wave'])
REST_GRID  = np.arange(np.floor(WMIN / (1 + Z_MAX)), WMAX, DLAM)
NORM_WIN   = (5400, 5600)
print(f'Rest grid: {REST_GRID[0]:.0f}–{REST_GRID[-1]:.0f} Å, {len(REST_GRID)} px')

# Redshift lookup from catalog
z_map = cat.drop_duplicates('name').set_index('name').nsa_z.to_dict()
tasks = [(i, p, n, z_map.get(n, np.nan)) for i, (p, n) in enumerate(zip(inv.path, inv.name))]
print(f'Files with a catalog redshift: {sum(np.isfinite(t[3]) for t in tasks):,} / {len(tasks):,}')

# ---- Spectral indices (rest-frame Å) ------------------------------------------
LINES = {  # name: (line window, blue continuum, red continuum)
    'Ha':     ((6553.0, 6573.0), (6490, 6530), (6600, 6640)),
    'NII6584':((6576.0, 6591.0), (6490, 6530), (6600, 6640)),
    'Hb':     ((4851.0, 4871.0), (4815, 4845), (4880, 4910)),
    'OIII5007':((4997.0, 5017.0), (4975, 4990), (5025, 5050)),
}
HDA = ((4083.50, 4122.25), (4041.60, 4079.75), (4128.50, 4161.00))   # Worthey & Ottaviani 1997

_trapz = getattr(np, "trapezoid", None) or np.trapz

def _mean(w, f, lo, hi):
    m = (w >= lo) & (w <= hi) & np.isfinite(f)
    return (np.mean(w[m]), np.mean(f[m]), m.sum()) if m.sum() >= 3 else (np.nan, np.nan, 0)

def equiv_width(w, f, line, blue, red, emission=True):
    wb, fb, nb = _mean(w, f, *blue); wr, fr, nr = _mean(w, f, *red)
    m = (w >= line[0]) & (w <= line[1]) & np.isfinite(f)
    if nb == 0 or nr == 0 or m.sum() < 3: return np.nan
    cont = fb + (fr - fb) * (w[m] - wb) / (wr - wb)
    if np.any(cont <= 0): return np.nan
    integrand = (f[m] - cont) / cont if emission else (1 - f[m] / cont)
    return _trapz(integrand, w[m])

def dn4000(w, f):
    fnu = f * w**2                                    # F_nu ∝ F_λ λ²  (Balogh+99 narrow definition)
    b = (w >= 3850) & (w <= 3950) & np.isfinite(fnu); r = (w >= 4000) & (w <= 4100) & np.isfinite(fnu)
    return np.mean(fnu[r]) / np.mean(fnu[b]) if b.sum() > 5 and r.sum() > 5 and np.mean(fnu[b]) > 0 else np.nan

def band_median(w, f, lo, hi):
    m = (w >= lo) & (w <= hi) & np.isfinite(f)
    return np.median(f[m]) if m.sum() > 5 else np.nan

In [ ]:
def process_one(task):
    i, path, name, z = task
    feat = {'idx': i, 'name': name, 'error': None}
    rest = {k: None for k in ('1re', 'fov')}
    try:
        s = read_spectrum(path)
        w = s['wave']
        dw = np.diff(w)
        feat.update(n_pix=len(w), wmin=np.nanmin(w), wmax=np.nanmax(w), dlam_med=np.nanmedian(dw),
                    wave_monotonic=bool(np.all(dw > 0)), wave_nan=int((~np.isfinite(w)).sum()))
        for k in ('1re', 'fov'):
            f, e = s[f'flux_{k}'], s[f'err_{k}']
            bad = ~np.isfinite(f) | ~np.isfinite(e) | (e <= 0)
            good = ~bad
            snr = f[good] / e[good]
            feat.update({
                f'frac_bad_{k}':  bad.mean(),
                f'frac_neg_{k}':  (f[good] < 0).mean() if good.any() else np.nan,
                f'flux_med_{k}':  np.median(f[good]) if good.any() else np.nan,
                f'err_med_{k}':   np.median(e[good]) if good.any() else np.nan,
                f'err_cv_{k}':    np.std(e[good]) / np.mean(e[good]) if good.any() else np.nan,  # ~0 → constant error array
                f'snr_med_{k}':   np.median(snr) if good.any() else np.nan,
                f'snr_p10_{k}':   np.percentile(snr, 10) if good.any() else np.nan,
                f'all_zero_{k}':  bool(np.all(f[np.isfinite(f)] == 0)),
            })
            if np.isfinite(z) and z > -0.01 and good.sum() > 100:
                wr = w / (1 + z); fr = np.where(good, f, np.nan)
                norm = band_median(wr, fr, *NORM_WIN)
                feat[f'norm_{k}'] = norm
                feat[f'Dn4000_{k}'] = dn4000(wr, fr)
                feat[f'HdA_{k}'] = equiv_width(wr, fr, *HDA, emission=False)
                for ln, (lw, bl, rd) in LINES.items():
                    feat[f'EW_{ln}_{k}'] = equiv_width(wr, fr, lw, bl, rd)
                blue, red = band_median(wr, fr, 4400, 4600), band_median(wr, fr, 6000, 6200)
                feat[f'color_{k}'] = -2.5 * np.log10(blue / red) if blue > 0 and red > 0 else np.nan
                # resample good pixels to the common rest grid; mask gaps and edges
                spec = np.interp(REST_GRID, wr[good], f[good], left=np.nan, right=np.nan)
                badfrac = np.interp(REST_GRID, wr, bad.astype(float))
                spec[badfrac > 0.5] = np.nan
                if np.isfinite(norm) and norm > 0:
                    rest[k] = (spec / norm).astype(np.float32)
    except Exception as ex:
        feat['error'] = f'{type(ex).__name__}: {ex}'
    return feat, rest['1re'], rest['fov']

In [ ]:
FEAT_PATH = f'{OUT_DIR}/spectral_features.parquet'
SPEC_PATHS = {k: f'{OUT_DIR}/rest_spectra_{k}.npy' for k in ('1re', 'fov')}

cached = (not FORCE_REBUILD and os.path.exists(FEAT_PATH) and all(map(os.path.exists, SPEC_PATHS.values())))
if cached:
    feats = pd.read_parquet(FEAT_PATH)
    cached = len(feats) == len(tasks) and (feats.name.values == inv.name.values).all()
    if not cached: print('Cache does not match current file list → rebuilding')

if not cached:
    local = {k: f'{WORK_DIR}/rest_spectra_{k}.npy' for k in SPEC_PATHS}
    mm = {k: np.lib.format.open_memmap(p, mode='w+', dtype=np.float32, shape=(len(tasks), len(REST_GRID)))
          for k, p in local.items()}
    for k in mm: mm[k][:] = np.nan
    rows = []
    t0 = time.time()
    with ProcessPoolExecutor(max_workers=N_WORKERS) as ex:
        for feat, sp1, spf in tqdm(ex.map(process_one, tasks, chunksize=32), total=len(tasks), desc='FITS'):
            rows.append(feat)
            if sp1 is not None: mm['1re'][feat['idx']] = sp1
            if spf is not None: mm['fov'][feat['idx']] = spf
    for m in mm.values(): m.flush()
    del mm
    feats = pd.DataFrame(rows).sort_values('idx').reset_index(drop=True)
    feats.to_parquet(FEAT_PATH)
    for k in SPEC_PATHS: shutil.copy(local[k], SPEC_PATHS[k])
    np.save(f'{OUT_DIR}/rest_grid.npy', REST_GRID)
    print(f'Processed {len(feats):,} files in {(time.time()-t0)/60:.1f} min')

SPEC = {k: np.load(p, mmap_mode='r') for k, p in SPEC_PATHS.items()}
REST_GRID = np.load(f'{OUT_DIR}/rest_grid.npy')
print(f"Read errors: {feats.error.notna().sum()}")
display(feats.error.value_counts().head(10))
display(feats.head())

## 7. Cross-match spectra ↔ catalog

In [ ]:
cat_u = cat.drop_duplicates('name')
in_files, in_cat = set(feats.name), set(cat_u.name)
print(f'FITS files                : {len(in_files):,}')
print(f'Catalog rows (unique name): {len(in_cat):,}')
print(f'Matched                   : {len(in_files & in_cat):,}')
print(f'FITS without catalog row  : {len(in_files - in_cat):,}  e.g. {sorted(in_files - in_cat)[:5]}')
print(f'Catalog rows without FITS : {len(in_cat - in_files):,}')

df = feats.merge(cat_u, on='name', how='left', validate='one_to_one')   # one row per FITS, same order as SPEC
df['in_catalog'] = df.objra.notna()

# Is the spectroscopic subset representative of the full catalog?
if len(in_cat - in_files) > 0:
    cat['has_fits'] = cat.name.isin(in_files)
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    for ax, c in zip(axes, ['nsa_z', 'LogMass', 'Ttype_clean']):
        sns.histplot(data=cat.sample(min(len(cat), PLOT_SAMPLE), random_state=1), x=c, hue='has_fits',
                     stat='density', common_norm=False, element='step', bins=40, ax=ax)
    plt.suptitle('Catalog: with vs without FITS (selection bias check)'); plt.tight_layout()
    savefig('08_selection_bias'); plt.show()

## 8. Spectral data-quality EDA
Things to look for: inconsistent wavelength grids, bad/negative pixels, very low S/N, and **error arrays that are almost constant**
(`err_cv ≈ 0`). In the sample file, `ERROR_FoV` stays at ~31.6 for every pixel while `FLUX_FoV` is ~20, which gives S/N < 1,
whereas `ERROR_1RE` varies with S/N ~100. Check whether that is expected for the FoV product before using FoV errors.

In [ ]:
print('Wavelength-grid configurations (n_pix, wmin, wmax, Δλ):')
display(df.groupby(['n_pix', df.wmin.round(1), df.wmax.round(1), df.dlam_med.round(3)]).size()
          .rename('n_files').sort_values(ascending=False).head(10))
print(f"Non-monotonic λ: {(~df.wave_monotonic.fillna(True)).sum()} | files with NaN λ: {(df.wave_nan > 0).sum()}")

q_cols = ['frac_bad', 'frac_neg', 'snr_med', 'err_cv']
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, q in zip(axes, q_cols):
    for k, c in zip(('1re', 'fov'), ('C0', 'C1')):
        v = df[f'{q}_{k}'].replace([np.inf, -np.inf], np.nan).dropna()
        if q in ('snr_med', 'err_cv'): v = np.log10(v[v > 0]); ax.set_xlabel(f'log10 {q}')
        else: ax.set_xlabel(q)
        ax.hist(v, bins=60, alpha=.6, color=c, label=k.upper())
    ax.legend(); ax.set_title(q)
plt.tight_layout(); savefig('09_quality_hist'); plt.show()

display(df[[c for c in df.columns if any(c.startswith(q) for q in q_cols)]].describe(percentiles=[.01, .5, .99]).T)

In [ ]:
# S/N drivers: redshift, mass, IFU size
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
m = df.in_catalog
axes[0].hexbin(df.loc[m, 'nsa_z'], np.log10(df.loc[m, 'snr_med_1re']), gridsize=50, bins='log', mincnt=1)
axes[0].set(xlabel='nsa_z', ylabel='log10 S/N (1RE)', title='S/N vs redshift')
axes[1].hexbin(df.loc[m, 'LogMass'], np.log10(df.loc[m, 'snr_med_1re']), gridsize=50, bins='log', mincnt=1)
axes[1].set(xlabel='LogMass', ylabel='log10 S/N (1RE)', title='S/N vs stellar mass')
df['flux_ratio_1re_fov'] = df.flux_med_1re / df.flux_med_fov
sns.boxplot(data=df[m], x='ifu_size', y='flux_ratio_1re_fov', showfliers=False, ax=axes[2])
axes[2].set_title('Aperture flux ratio 1Re / FoV by IFU size')
plt.tight_layout(); savefig('10_snr_drivers'); plt.show()

In [ ]:
# Gallery: random spectra + the worst ones
def plot_gallery(idx, title, n=8, key='1re'):
    idx = list(idx)[:n]
    fig, axes = plt.subplots(len(idx), 1, figsize=(13, 1.6 * len(idx)), sharex=True)
    for ax, i in zip(np.atleast_1d(axes), idx):
        ax.plot(REST_GRID, SPEC[key][i], lw=.5)
        r = df.loc[i]
        ax.text(.01, .75, f"{r['name']} | {r.get('Type', '')} | z={r.get('nsa_z', np.nan):.3f} | "
                f"S/N={r[f'snr_med_{key}']:.1f} | bad={r[f'frac_bad_{key}']:.2%}", transform=ax.transAxes, fontsize=8)
    np.atleast_1d(axes)[-1].set_xlabel('Rest wavelength [Å]'); fig.suptitle(title, y=1.0)
    plt.tight_layout(); return fig

has_spec = np.isfinite(np.asarray(SPEC['1re'][:, len(REST_GRID) // 2]))
rng = np.random.default_rng(RANDOM_STATE)
plot_gallery(rng.choice(np.where(has_spec)[0], 8, replace=False), 'Random spectra (normalised, rest frame)')
savefig('11_gallery_random'); plt.show()
plot_gallery(df[has_spec].nsmallest(8, 'snr_med_1re').index, 'Lowest-S/N spectra'); savefig('12_gallery_low_snr'); plt.show()

## 9. Spectra vs. morphology and physical properties

In [ ]:
# Median stacked spectra by morphological class, with zooms on key features
ZOOMS = [('Full', (REST_GRID[0], REST_GRID[-1])), ('4000 Å break', (3700, 4400)),
         ('Hβ + [OIII]', (4800, 5050)), ('Hα + [NII]', (6500, 6620))]
fig, axes = plt.subplots(2, 2, figsize=(16, 8)); axes = axes.ravel()
stack_counts = {}
for j, cls in enumerate(MORPH_ORDER):
    idx = np.where((df.morph_class == cls).values & has_spec)[0]
    if len(idx) < 5: continue
    sel = rng.choice(idx, min(len(idx), 3000), replace=False)        # cap for RAM
    stack = np.nanmedian(np.asarray(SPEC['1re'][np.sort(sel)]), axis=0)
    stack_counts[cls] = len(idx)
    for ax, (ttl, (lo, hi)) in zip(axes, ZOOMS):
        w = (REST_GRID >= lo) & (REST_GRID <= hi)
        ax.plot(REST_GRID[w], stack[w], lw=.8, label=f'{cls} (n={len(idx)})', color=f'C{j}')
        ax.set_title(ttl)
axes[0].legend(); [ax.set_xlabel('Rest λ [Å]') for ax in axes]
plt.tight_layout(); savefig('13_stacks_by_morphology'); plt.show()

In [ ]:
# Classic diagnostics
d = df[df.in_catalog].copy()
fig, axes = plt.subplots(2, 2, figsize=(15, 11))

sc = axes[0, 0].scatter(d.LogMass, d.Dn4000_1re, c=d.Ttype_clean, cmap='coolwarm_r', s=4, alpha=.6)
axes[0, 0].set(xlabel='LogMass', ylabel='Dn4000 (1Re)', ylim=(0.8, 2.4), title='Dn4000 vs mass (red sequence / blue cloud)')
plt.colorbar(sc, ax=axes[0, 0], label='T-type')

axes[0, 1].scatter(d.Dn4000_1re, d.HdA_1re, c=d.Ttype_clean, cmap='coolwarm_r', s=4, alpha=.6)
axes[0, 1].set(xlabel='Dn4000', ylabel='HδA [Å]', xlim=(0.8, 2.4), ylim=(-6, 12), title='Star-formation history plane')

# WHAN diagram (EW ratio used as proxy for the flux ratio — same continuum)
x = np.log10(d.EW_NII6584_1re / d.EW_Ha_1re); y = np.log10(d.EW_Ha_1re)
ok = np.isfinite(x) & np.isfinite(y)
axes[1, 0].hexbin(x[ok], y[ok], gridsize=60, bins='log', mincnt=1, extent=(-1.5, 1, -1, 3))
axes[1, 0].axvline(-0.4, c='k', lw=.8); axes[1, 0].axhline(np.log10(3), c='k', lw=.8); axes[1, 0].axhline(np.log10(6), c='k', ls='--', lw=.8)
axes[1, 0].set(xlabel='log [NII]/Hα', ylabel='log EW(Hα) [Å]', title='WHAN (approx.): SF | AGN | retired')

# BPT (approx. via EW ratios) with Kauffmann+03 line
x2 = np.log10(d.EW_NII6584_1re / d.EW_Ha_1re); y2 = np.log10(d.EW_OIII5007_1re / d.EW_Hb_1re)
ok2 = np.isfinite(x2) & np.isfinite(y2) & (d.EW_Hb_1re > 1) & (d.EW_Ha_1re > 1)
axes[1, 1].scatter(x2[ok2], y2[ok2], c=d.Ttype_clean[ok2], cmap='coolwarm_r', s=4, alpha=.6)
xx = np.linspace(-1.5, 0.0, 100); axes[1, 1].plot(xx, 0.61 / (xx - 0.05) + 1.3, 'k-', lw=1)
axes[1, 1].set(xlim=(-1.5, 0.6), ylim=(-1.2, 1.3), xlabel='log [NII]/Hα', ylabel='log [OIII]/Hβ', title='BPT (approx.; EW>1 Å)')
plt.tight_layout(); savefig('14_diagnostics'); plt.show()

In [ ]:
# Spectral features by morphology & dust/inclination effect in spirals
feat_cols = ['Dn4000_1re', 'HdA_1re', 'EW_Ha_1re', 'color_1re']
fig, axes = plt.subplots(1, 4, figsize=(19, 4.5))
for ax, c in zip(axes, feat_cols):
    sns.boxplot(data=d, x='morph_class', y=c, order=MORPH_ORDER, showfliers=False, ax=ax)
    ax.tick_params(axis='x', rotation=45)
    if c == 'EW_Ha_1re': ax.set_yscale('symlog')
plt.tight_layout(); savefig('15_features_by_morph'); plt.show()

sp = d[d.Ttype_clean >= 1]
fig, ax = plt.subplots(figsize=(7, 4.5))
sns.regplot(x=sp.incl, y=sp.color_1re, scatter_kws=dict(s=3, alpha=.3), line_kws=dict(color='r'), lowess=True, ax=ax)
ax.set(title='Spirals: colour vs inclination (dust reddening)', xlabel='incl [deg]', ylabel='colour proxy (6100/4500 Å) [mag]')
savefig('16_color_vs_incl'); plt.show()

In [ ]:
# Spearman correlation: catalog × spectral features
corr_cols = ['nsa_z', 'LogMass', 'Ttype_clean', 'incl', 'PETRO_TH90', 'R90_kpc', 'ifu_size',
             'snr_med_1re', 'frac_bad_1re', 'Dn4000_1re', 'HdA_1re', 'EW_Ha_1re', 'EW_OIII5007_1re', 'color_1re',
             'flux_ratio_1re_fov']
C = d[corr_cols].astype(float).corr(method='spearman')
plt.figure(figsize=(11, 9))
sns.heatmap(C, cmap='RdBu_r', center=0, vmin=-1, vmax=1, annot=True, fmt='.2f', annot_kws={'size': 7}, square=True)
plt.title('Spearman correlations'); savefig('17_correlations'); plt.show()

# 1Re vs FoV: does the aperture change the stellar population signal?
fig, ax = plt.subplots(figsize=(5.5, 5))
ax.hexbin(d.Dn4000_1re, d.Dn4000_fov, gridsize=50, bins='log', mincnt=1, extent=(0.8, 2.4, 0.8, 2.4))
ax.plot([0.8, 2.4], [0.8, 2.4], 'r--'); ax.set(xlabel='Dn4000 1Re', ylabel='Dn4000 FoV', title='Aperture effect')
savefig('18_aperture_dn4000'); plt.show()

## 10. Dimensionality reduction (PCA + UMAP)
PCA runs on the rest-frame wavelength range that ≥ 98% of spectra cover. Small gaps are filled by linear interpolation.

In [ ]:
from sklearn.decomposition import PCA
X_all = np.asarray(SPEC['1re'])
cov = np.isfinite(X_all[has_spec]).mean(0)
wsel = cov >= 0.98
print(f'PCA range: {REST_GRID[wsel].min():.0f}–{REST_GRID[wsel].max():.0f} Å ({wsel.sum()} px)')

X = X_all[:, wsel].copy()
rows = has_spec & (np.isfinite(X).mean(1) > 0.95)
X = X[rows]
for r in np.where(~np.isfinite(X).all(1))[0]:            # fill small gaps
    g = np.isfinite(X[r]); X[r, ~g] = np.interp(np.where(~g)[0], np.where(g)[0], X[r, g])
clip = np.nanpercentile(X, [0.1, 99.9]); X = np.clip(X, *clip)   # tame cosmic rays / sky residuals
pca_rows = np.where(rows)[0]
print(f'Spectra in PCA: {len(pca_rows):,}')

N_PC = 20
pca = PCA(N_PC, svd_solver='randomized', random_state=RANDOM_STATE).fit(X)
P = pca.transform(X)
for j in range(N_PC): df.loc[pca_rows, f'PC{j+1}'] = P[:, j]

fig, axes = plt.subplots(1, 2, figsize=(16, 4.5), gridspec_kw={'width_ratios': [1, 2.5]})
axes[0].plot(np.arange(1, N_PC + 1), np.cumsum(pca.explained_variance_ratio_), 'o-')
axes[0].set(xlabel='# components', ylabel='cumulative explained variance', title='Scree')
for j in range(4):
    axes[1].plot(REST_GRID[wsel], pca.components_[j] + 0.08 * j, lw=.7, label=f'PC{j+1} ({pca.explained_variance_ratio_[j]:.1%})')
axes[1].plot(REST_GRID[wsel], pca.mean_ * 0.05 - 0.08, 'k', lw=.7, label='mean (scaled)')
axes[1].legend(ncol=5, fontsize=8); axes[1].set(xlabel='Rest λ [Å]', title='Eigenspectra (offset)')
plt.tight_layout(); savefig('19_pca'); plt.show()

In [ ]:
dp = df.loc[pca_rows]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, c, cm in zip(axes, ['Ttype_clean', 'Dn4000_1re', 'LogMass'], ['coolwarm_r', 'viridis', 'magma']):
    s = ax.scatter(dp.PC1, dp.PC2, c=dp[c], s=3, cmap=cm, alpha=.7,
                   vmin=dp[c].quantile(.02), vmax=dp[c].quantile(.98))
    plt.colorbar(s, ax=ax, label=c); ax.set(xlabel='PC1', ylabel='PC2', title=f'PCA coloured by {c}')
plt.tight_layout(); savefig('20_pc1_pc2'); plt.show()

# How well does the spectrum predict morphology? (quick baseline)
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
lab = dp.morph_class.astype(str); okl = lab != 'Unclassified'
if okl.sum() > 100 and lab[okl].nunique() > 1:
    rf = RandomForestClassifier(300, n_jobs=-1, random_state=RANDOM_STATE, class_weight='balanced')
    acc = cross_val_score(rf, P[okl.values], lab[okl], cv=5, scoring='balanced_accuracy')
    print(f'Morph class from 20 PCs → balanced accuracy {acc.mean():.3f} ± {acc.std():.3f} '
          f'(chance ≈ {1/lab[okl].nunique():.2f})')

In [ ]:
try:
    import umap
    emb = umap.UMAP(n_neighbors=30, min_dist=0.1, random_state=RANDOM_STATE).fit_transform(P)
    df.loc[pca_rows, ['UMAP1', 'UMAP2']] = emb
    fig, axes = plt.subplots(1, 2, figsize=(16, 6.5))
    sns.scatterplot(x=emb[:, 0], y=emb[:, 1], hue=dp.morph_class.astype(str).values, hue_order=MORPH_ORDER,
                    s=4, linewidth=0, ax=axes[0]); axes[0].set_title('UMAP of spectra — morphology')
    s = axes[1].scatter(emb[:, 0], emb[:, 1], c=np.log10(dp.EW_Ha_1re.clip(lower=0.1)), s=4, cmap='Spectral_r')
    plt.colorbar(s, ax=axes[1], label='log EW(Hα)'); axes[1].set_title('UMAP — star-formation activity')
    savefig('21_umap'); plt.show()
except ImportError:
    print('umap-learn not installed — skipping')

## 11. Anomaly detection

In [ ]:
from sklearn.ensemble import IsolationForest
K = 10
recon = P[:, :K] @ pca.components_[:K] + pca.mean_
df.loc[pca_rows, 'pca_recon_err'] = np.sqrt(np.mean((X - recon) ** 2, axis=1))

tab_cols = ['snr_med_1re', 'frac_bad_1re', 'Dn4000_1re', 'HdA_1re', 'EW_Ha_1re', 'color_1re', 'flux_ratio_1re_fov']
T = df.loc[pca_rows, tab_cols].astype(float)
T = T.fillna(T.median())
iso = IsolationForest(n_estimators=300, contamination=0.01, random_state=RANDOM_STATE).fit(T)
df.loc[pca_rows, 'iso_score'] = -iso.score_samples(T)

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(np.log10(df.pca_recon_err.dropna()), bins=80); ax.set(xlabel='log10 PCA reconstruction RMS', ylabel='N', title='Spectral weirdness')
savefig('22_recon_err'); plt.show()

plot_gallery(df.nlargest(8, 'pca_recon_err').index, f'Top anomalies: PCA reconstruction error (K={K})')
savefig('23_anomalies_pca'); plt.show()
plot_gallery(df.nlargest(8, 'iso_score').index, 'Top anomalies: Isolation Forest on features')
savefig('24_anomalies_iso'); plt.show()

## 12. Data-quality report & exports

In [ ]:
SNR_MIN, BAD_MAX = 5, 0.20
flags = pd.DataFrame(index=df.index)
flags['read_error']          = df.error.notna()
flags['not_in_catalog']      = ~df.in_catalog
flags['no_redshift']         = df.in_catalog & df.nsa_z.isna()
flags['no_rest_spectrum']    = ~has_spec
flags['low_snr_1re']         = df.snr_med_1re < SNR_MIN
flags['many_bad_pix_1re']    = df.frac_bad_1re > BAD_MAX
flags['all_zero_flux']       = df.all_zero_1re.fillna(False).astype(bool) | df.all_zero_fov.fillna(False).astype(bool)
flags['const_error_fov']     = df.err_cv_fov < 1e-2
flags['const_error_1re']     = df.err_cv_1re < 1e-2
flags['incl_zero']           = df.incl == 0
flags['ttype_special']       = df.ttype_special.fillna(False).astype(bool)
flags['pca_outlier']         = df.pca_recon_err > df.pca_recon_err.quantile(0.99)
flags['iso_outlier']         = df.iso_score > df.iso_score.quantile(0.99)

report = pd.DataFrame({'n': flags.sum(), 'pct': (100 * flags.mean()).round(2)}).sort_values('n', ascending=False)
display(report)
df['n_flags'] = flags.sum(1)
df['clean'] = ~flags[['read_error', 'not_in_catalog', 'no_redshift', 'no_rest_spectrum',
                      'low_snr_1re', 'many_bad_pix_1re', 'all_zero_flux']].any(axis=1)
print(f"Clean sample for science: {df.clean.sum():,} / {len(df):,}")

# ---- Exports ----
df.drop(columns=['path'], errors='ignore').to_parquet(f'{OUT_DIR}/merged_catalog_spectra.parquet')
pd.concat([df[['name']], flags], axis=1).to_csv(f'{OUT_DIR}/quality_flags.csv', index=False)
df.loc[flags.pca_outlier | flags.iso_outlier,
       ['name', 'Type', 'nsa_z', 'snr_med_1re', 'pca_recon_err', 'iso_score']].sort_values('pca_recon_err', ascending=False) \
  .to_csv(f'{OUT_DIR}/anomaly_candidates.csv', index=False)
qc_cat.to_csv(f'{OUT_DIR}/catalog_qc.csv')
report.to_csv(f'{OUT_DIR}/spectra_qc.csv')
print('Saved to', OUT_DIR); print(*sorted(os.listdir(OUT_DIR)), sep='\n')

### Next steps
* Reload later with `pd.read_parquet(f'{OUT_DIR}/merged_catalog_spectra.parquet')` and `np.load(f'{OUT_DIR}/rest_spectra_1re.npy', mmap_mode='r')`. You won't need to re-read the FITS files.
* Line indices here are simple EW windows meant for EDA. For science-grade line fluxes, use full spectral fitting (e.g. pPXF) or MaNGA DAP products.
* Use the `clean` column and `quality_flags.csv` to define the analysis sample. Inspect `anomaly_candidates.csv` by eye.